# Case study · Berlin weather from a web API

*Session 2, block 2 practice. Author: course team, licence CC-BY-4.0. Weather data: Open-Meteo, CC BY 4.0.*

**The question.** Does the weather explain how busy Berlin's Airbnb market is? Guests may travel more in warm, sunny months and stay away when it rains. The listings and their monthly reviews (the course's measure of demand) come as files from Inside Airbnb. The weather does not: it changes every day, and its provider, [Open-Meteo](https://open-meteo.com/), offers it through a **web API**. You ask for the place, the days and the variables you need, and the server answers with JSON.

In this notebook you

1. send a first request and read the answer (status, headers, JSON),
2. see what an error looks like and how the client reports it,
3. respect the rate limits and fetch ten years one year at a time,
4. parse every day into the validated class `DailyWeather` of the workspace,
5. take a first look at weather and demand (Session 12 does this properly in a forecast),
6. test the parsing offline with `httpx.MockTransport`.

The cells marked *live* need internet access. Open-Meteo is free for non-commercial use without a key, with fewer than 600 calls per minute, 5,000 per hour and 10,000 per day.

## Setup

In [ ]:
import sys
from datetime import date
from pathlib import Path

import httpx
import matplotlib.pyplot as plt
import pandas as pd

# The notebook may be started from the repository root or from its own folder.
here = Path.cwd().resolve()
root = next(p for p in [here, *here.parents] if (p / "case-study").is_dir())
sys.path.insert(0, str(root / "sessions" / "02-software-engineering" / "workspace" / "src"))

from listingtools import APIError, DailyWeather, WeatherClient  # noqa: E402
from listingtools.weather import ARCHIVE_URL, parse_daily  # noqa: E402

DATA = root / "case-study" / "data" / "airbnb"
print(ARCHIVE_URL)

## 1 · A first request (live)

An HTTP `GET` request consists of the endpoint URL and **query parameters**. httpx builds the part after `?` from a dictionary. Always set a **timeout**, so that the program does not hang if the server does not answer.

In [ ]:
params = {
    "latitude": 52.52, "longitude": 13.41,           # centre of Berlin
    "start_date": "2024-07-01", "end_date": "2024-07-07",
    "daily": "temperature_2m_mean,precipitation_sum,sunshine_duration",
    "timezone": "Europe/Berlin",
}
response = httpx.get(ARCHIVE_URL, params=params, timeout=10)
print(response.status_code, response.headers["content-type"])
print(response.url)
payload = response.json()
print(list(payload))
payload["daily_units"]

The table comes **by column**: one array of dates and one array per variable. Sunshine is given in seconds. A DataFrame turns the columns into a table:

In [ ]:
week = pd.DataFrame(payload["daily"])
week["sunshine_hours"] = (week.pop("sunshine_duration") / 3600).round(1)
week

**Exercise.** Change `daily` to also request `rain_sum` and `wind_speed_10m_max` (see the [documentation](https://open-meteo.com/en/docs/historical-weather-api)). Which units does `daily_units` report for them?

## 2 · When the request is wrong (live)

A wrong request is answered with status **400** and a JSON body that names the reason. Retrying does not help: the request has to be fixed.

In [ ]:
bad = httpx.get(ARCHIVE_URL, params={**params, "latitude": 152.52}, timeout=10)
print(bad.status_code, bad.json())

too_early = httpx.get(ARCHIVE_URL, params={**params, "start_date": "1900-01-01"}, timeout=10)
print(too_early.status_code, too_early.json()["reason"])

The workspace client turns such an answer into an `APIError` that carries the server's reason and the status code, and does not retry it:

In [ ]:
try:
    with WeatherClient(latitude=152.52) as broken:
        broken.daily(date(2024, 7, 1), date(2024, 7, 7))
except APIError as err:
    print(type(err).__name__, err.status_code, "|", err)

## 3 · Rate limits and paging by year

Open-Meteo counts a request for more than two weeks of data as several calls: a request costs about *days / 14* calls (and more if it asks for more than 10 variables). Ten years of daily weather are therefore not one call but almost 300, still far below the limit of 10,000 per day. Being polite to a free service is part of the job: request what you need once, store it, and do not re-download it in every notebook run.

Instead of one request for ten years, the client asks for one calendar year at a time. Each answer stays small, you see progress, and if one request fails, you lose one year, not ten. This is the same idea as **pagination**, where an API returns a long result page by page.

In [ ]:
def api_calls(days: int, n_variables: int = 3) -> float:
    """Calls counted by Open-Meteo for one request (fractional counting)."""
    return max(1.0, days / 14) * max(1.0, n_variables / 10)


print(round(api_calls(7), 1), round(api_calls(366), 1), round(api_calls(3830), 1))

In [ ]:
# live: ten and a half years, one request per year
start, end = date(2016, 1, 1), date(2026, 6, 26)        # the period of the course data
weather = []
skipped = 0
with WeatherClient() as client:
    for year in range(start.year, end.year + 1):
        chunk = client.daily(max(start, date(year, 1, 1)), min(end, date(year, 12, 31)))
        weather.extend(chunk.records)
        skipped += chunk.skipped
        print(year, len(chunk), "days")
print(len(weather), "days in total;", skipped, "skipped")

Workspace exercise 4 turns this loop into the method `WeatherClient.history(start, end)`, with a helper `yearly_chunks` that you can test without a network.

## 4 · Validated records

`client.daily()` already parsed every day into a `DailyWeather` object: a dataclass whose `__post_init__` checks that the date is a date, the temperature lies between −40 and 45 °C, rain is not negative and sunshine lies between 0 and 24 hours. A day with a missing value (`null` in the JSON, which happens for the most recent days of the archive) is skipped and counted instead of stopping the download.

In [ ]:
print(weather[0])
try:
    DailyWeather("2024-07-01", temperature=61.0, precipitation=0.0, sunshine_hours=10.0)
except ValueError as err:
    print("rejected:", err)

daily = pd.DataFrame([d.to_dict() for d in weather])
daily["date"] = pd.to_datetime(daily["date"])
daily.describe().round(1)

**Same data as the course file.** The preparation script `case-study/prepare_airbnb.py` fetched the same archive once and stored it as `weather_daily.parquet`. Session 12 reads that file, so the forecast does not depend on the API during class. Check that both agree:

In [ ]:
stored = pd.read_parquet(DATA / "weather_daily.parquet")
both = daily.merge(stored, on="date")
print(len(both), "days in both")
print("largest difference in temperature:", (both["temperature"] - both["temperature_2m_mean"]).abs().max())
print("largest difference in rain:", (both["precipitation"] - both["precipitation_sum"]).abs().max())

Small differences are possible: the archive is a *reanalysis* that the provider occasionally recomputes. Record the download date when you use API data in a report.

## 5 · A first look: weather and demand

Demand is measured by the number of reviews per month, summed over all listings in the snapshot. Weather is summarised per month: mean temperature, total rain and total hours of sunshine.

In [ ]:
reviews = pd.read_parquet(DATA / "reviews_monthly.parquet")
demand = reviews.groupby("month")["n_reviews"].sum().rename("reviews")

monthly = (daily.set_index("date")
           .resample("MS")
           .agg({"temperature": "mean", "precipitation": "sum", "sunshine_hours": "sum"})
           .join(demand, how="inner"))
monthly = monthly[monthly.index < "2026-06-01"]          # June 2026 is incomplete
monthly.tail()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8), layout="constrained")
axes[0].plot(monthly.index, monthly["reviews"], color="#2d7d74")
axes[0].set(title="Reviews per month", ylabel="reviews")
recent = monthly[monthly.index >= "2023-01-01"]
axes[1].scatter(recent["temperature"], recent["reviews"], color="#2d7d74")
axes[1].set(title="2023 to May 2026", xlabel="mean temperature of the month (°C)", ylabel="reviews")
for ax in axes:
    ax.spines[["top", "right"]].set_visible(False)
plt.show()

print("correlation with reviews, 2023 to May 2026:")
recent.corr()["reviews"].drop("reviews").round(2)

**Reading the result honestly.** Since 2023, warm and sunny months have clearly more reviews (correlation about 0.6), while rain shows almost no relation. But this does not show that the weather *causes* the demand: summer is also the season of school holidays, festivals and conferences, so temperature and season move together. The left chart also shows the COVID break of 2020–2021 and the growth since, which any comparison across years must take into account. Session 12 asks the sharper question: does the weather improve a demand forecast *beyond* the season?

**Exercise.** Compute the same correlations for 2016–2019. Are they weaker or stronger, and why could the share of listings that still exist in the snapshot matter? (Reviews are only available for listings that are still online in June 2026.)

## 6 · Testing the parsing offline

Tests must not depend on the network: the server may be down, slow or rate-limited, and continuous integration runs without internet access. `httpx.MockTransport` replaces the network with a function that plays the server. Here it answers with one valid and one incomplete day:

In [ ]:
def fake_server(request: httpx.Request) -> httpx.Response:
    assert request.url.params["start_date"] == "2026-06-25"
    return httpx.Response(200, json={"daily": {
        "time": ["2026-06-25", "2026-06-26"],
        "temperature_2m_mean": [28.6, None],                  # the last day is not yet available
        "precipitation_sum": [0.0, None],
        "sunshine_duration": [57600.0, None],
    }})


with WeatherClient(transport=httpx.MockTransport(fake_server)) as offline:
    series = offline.daily(date(2026, 6, 25), date(2026, 6, 26))
print(len(series), "valid,", series.skipped, "skipped:", series.records[0])
assert series.skipped == 1

The same pattern runs in the workspace tests (`tests/test_weather.py`), together with tests for `429` (wait and retry), `503` and timeouts (retry with growing waits), `400` (report the reason, no retry) and an HTML page instead of JSON.

`parse_daily` can also be tested without any client at all, because it takes a plain dictionary:

In [ ]:
parse_daily({"daily": {"time": ["2024-07-01"], "temperature_2m_mean": [17.6],
                       "precipitation_sum": [7.9], "sunshine_duration": [33378.58]}})

## Optional extra · Live departures in Berlin (VBB)

*Not part of the tests or the exercises.* The community-run API `v6.vbb.transport.rest` returns the departures of Berlin's public transport as **nested JSON**: a departure contains a line, a stop and a delay. It is run by volunteers and allows 100 requests per minute per IP address, so use it sparingly. Stop `900100003` is S+U Alexanderplatz.

In [ ]:
try:
    answer = httpx.get("https://v6.vbb.transport.rest/stops/900100003/departures",
                       params={"duration": 10, "results": 5}, timeout=10)
    answer.raise_for_status()
    departures = answer.json()["departures"]
    print(pd.DataFrame([{"line": d["line"]["name"], "direction": d["direction"],
                         "planned": d["plannedWhen"], "delay_min": (d["delay"] or 0) / 60}
                        for d in departures]))
except (httpx.HTTPError, KeyError, ValueError) as err:   # community service: may be unavailable
    print("VBB API not available:", err)

**Exercise (optional).** Which fields are nested inside `d["line"]` and `d["stop"]`? Write a small `Departure` dataclass that validates the fields you need.

## Next step

Block 3 of the session puts the workspace under version control and adds a feature module through a reviewed pull request. Session 3 loads the listings, calendar, reviews and `weather_daily` into a database and joins them with SQL.